In [1]:
# ============================================================
# POINT 4:
# Customers most likely to increase spending
# if inventory improves for top-selling products
# Dataset: final_clean_dataset.xls  (actually CSV content)
# ============================================================

import pandas as pd
import numpy as np

# 1. Load final clean dataset
df = pd.read_csv("final_clean_dataset.xls")   # file is CSV with .xls extension

print("Data loaded ✔")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

# ------------------------------------------------------------
# 2. Identify TOP-SELLING products (by quantity + revenue)
# ------------------------------------------------------------
prod_perf = (
    df.groupby(["product_id", "product_name", "product_category"], as_index=False)
      .agg(
          qty_sold  = ("quantity", "sum"),
          rev_sold  = ("line_revenue", "sum"),
          avg_price = ("unit_price", "mean"),
          stock_now = ("current_stock_level", "mean")
      )
)

# Define "top-selling" as top 10% by quantity sold
q90_qty = prod_perf["qty_sold"].quantile(0.90)
prod_perf["is_top_seller"] = prod_perf["qty_sold"] >= q90_qty

print("\nTop-selling products (sample):")
display(
    prod_perf[prod_perf["is_top_seller"]]
    .sort_values("qty_sold", ascending=False)
    .head(10)
)

# ------------------------------------------------------------
# 3. Among top-sellers, find LOW-INVENTORY items
# ------------------------------------------------------------
top_sellers = prod_perf[prod_perf["is_top_seller"]].copy()

# Low inventory = bottom 30% of stock among top-sellers
q30_stock = top_sellers["stock_now"].quantile(0.30)
top_sellers["is_low_stock"] = top_sellers["stock_now"] <= q30_stock

# Inventory-constrained top-selling products
top_sellers["is_constrained_top_product"] = top_sellers["is_low_stock"]

print("\nInventory-constrained top-selling products:")
display(
    top_sellers[top_sellers["is_constrained_top_product"]]
    .sort_values("qty_sold", ascending=False)
    .head(15)
)

# ------------------------------------------------------------
# 4. Tag these products back onto the transaction-level data
# ------------------------------------------------------------
df = df.merge(
    top_sellers[["product_id", "is_constrained_top_product"]],
    on="product_id",
    how="left"
)

df["is_constrained_top_product"] = df["is_constrained_top_product"].fillna(False)

print(
    "\n% of line-items that are top-selling & low-stock:",
    round(100 * df["is_constrained_top_product"].mean(), 2), "%"
)

# ------------------------------------------------------------
# 5. Customer-level metrics
# ------------------------------------------------------------

# Overall customer behaviour
cust_all = (
    df.groupby("customer_id", as_index=False)
      .agg(
          total_spend   = ("line_revenue", "sum"),
          total_orders  = ("transaction_id", "nunique"),
          total_items   = ("line_item_id", "count"),
      )
)

# Behaviour specifically on constrained top-selling products
cust_constrained = (
    df[df["is_constrained_top_product"]]
      .groupby("customer_id", as_index=False)
      .agg(
          constrained_spend   = ("line_revenue", "sum"),
          constrained_orders  = ("transaction_id", "nunique"),
          constrained_items   = ("line_item_id", "count"),
      )
)

# Merge
cust = cust_all.merge(cust_constrained, on="customer_id", how="left").fillna(0)

# ------------------------------------------------------------
# 6. Build an "UPLIFT SCORE" per customer
# ------------------------------------------------------------

# Share of spend on constrained top-selling products
cust["constrained_share"] = np.where(
    cust["total_spend"] > 0,
    cust["constrained_spend"] / cust["total_spend"],
    0
)

# Average ticket size
cust["avg_ticket"] = np.where(
    cust["total_orders"] > 0,
    cust["total_spend"] / cust["total_orders"],
    0
)

# Uplift score combines:
# - how focused they are on constrained top products
# - how many orders include those products
# - how big their usual basket is
cust["uplift_score"] = (
    cust["constrained_share"] *
    np.log1p(cust["constrained_orders"]) *
    cust["avg_ticket"]
)

# ------------------------------------------------------------
# 7. Final list: customers most likely to increase spending
# ------------------------------------------------------------

top_customers = (
    cust[cust["constrained_orders"] > 0]             # they actually buy those items
      .sort_values("uplift_score", ascending=False)
      .head(20)
)

print("\nTop 20 customers likely to increase spend if inventory improves:")
display(top_customers)

# ------------------------------------------------------------
# 8. Save outputs for PPT / Dashboard
# ------------------------------------------------------------
cust.to_csv("customer_uplift_potential.csv", index=False)
top_customers.to_csv("top_uplift_customers.csv", index=False)

print("\nFiles saved:")
print(" - customer_uplift_potential.csv  (all customers with uplift_score)")
print(" - top_uplift_customers.csv       (top 20 high-potential customers)") 


Data loaded ✔
Shape: (5000, 25)
Columns: ['line_item_id', 'transaction_id', 'product_id', 'promotion_id', 'quantity', 'line_item_amount', 'customer_id', 'store_id', 'transaction_date', 'total_amount', 'customer_phone', 'product_name', 'product_category', 'unit_price', 'current_stock_level', 'store_name', 'store_city', 'store_region', 'opening_date', 'line_revenue', 'got_promo', 'transaction_date_missing', 'year', 'month', 'day']


,line_item_id,transaction_id,product_id,promotion_id,quantity,line_item_amount,customer_id,store_id,transaction_date,total_amount,...,store_name,store_city,store_region,opening_date,line_revenue,got_promo,transaction_date_missing,year,month,day
0,1,T002404,P00534,PR034,3,1193.29,C04671,S00212,2024-01-09,1043.35,...,Costa Inc Store,Kolkata,West,2016-10-06,7571.49,1,1,2024,2024-01,Tuesday
1,2,T001639,P02463,PR012,8,1101.76,C00360,S00423,2024-01-09,1147.42,...,Wiggins Group Store,Ahmedabad,North,2022-12-04,30284.08,1,0,2024,2024-01,Tuesday
2,3,T002566,P01225,NO_PROMO,8,3035.35,C02529,S02612,2024-01-09,1137.52,...,"Espinoza, Simpson and Lewis Store",Delhi,South,2021-09-25,22534.96,1,1,2024,2024-01,Tuesday
3,4,T003125,P02581,PR002,10,1509.38,C03737,S03219,2024-02-03,827.79,...,"Cordova, Clark and Davis Store",Delhi,North,2019-07-30,27261.80,1,0,2024,2024-02,Saturday
4,5,T001304,P02551,PR014,9,2401.73,C00669,S02434,2024-02-03,752.71,...,Castro-Ramos Store,Ahmedabad,North,2017-11-06,25172.91,1,1,2024,2024-02,Saturday



Top-selling products (sample):


,product_id,product_name,product_category,qty_sold,rev_sold,avg_price,stock_now,is_top_seller
2120,P02590,Fire Lite,Beauty,50,114231.50,2284.63,250.0,True
1096,P01355,Word Max,Apparel,48,172018.08,3583.71,493.0,True
391,P00492,Stuff Plus,Automotive,47,30259.07,643.81,5.0,True
199,P00245,Various Pro,Books,43,93105.32,2165.24,496.0,True
678,P00831,A Max,Home & Kitchen,40,9438.80,235.97,158.0,True
372,P00471,Ever Pro,Automotive,39,100353.63,2573.17,490.0,True
1337,P01650,Human Max,Beauty,39,43754.88,1121.92,120.0,True
971,P01207,Under Max,Apparel,37,146144.82,3949.86,227.0,True
2189,P02675,Especially Lite,Sports,37,163808.99,4427.27,407.0,True
1342,P01655,Drop Pro,Grocery,36,143114.04,3975.39,19.0,True



Inventory-constrained top-selling products:


,product_id,product_name,product_category,qty_sold,rev_sold,avg_price,stock_now,is_top_seller,is_low_stock,is_constrained_top_product
391,P00492,Stuff Plus,Automotive,47,30259.07,643.81,5.0,True,True,True
678,P00831,A Max,Home & Kitchen,40,9438.80,235.97,158.0,True,True,True
1337,P01650,Human Max,Beauty,39,43754.88,1121.92,120.0,True,True,True
1342,P01655,Drop Pro,Grocery,36,143114.04,3975.39,19.0,True,True,True
1973,P02411,But Max,Grocery,35,136954.30,3912.98,159.0,True,True,True
1057,P01312,National Plus,Grocery,34,102260.44,3007.66,34.0,True,True,True
171,P00211,View Max,Apparel,34,162720.94,4785.91,30.0,True,True,True
24,P00029,Method Lite,Automotive,33,63317.76,1918.72,114.0,True,True,True
1295,P01596,Grow Edition,Electronics,33,22667.37,686.89,1.0,True,True,True
1523,P01881,New Edition,Home & Kitchen,32,48428.80,1513.40,163.0,True,True,True


<ipython-input-1-8a2245b9f048>:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_constrained_top_product"] = df["is_constrained_top_product"].fillna(False)



% of line-items that are top-selling & low-stock: 6.24 %

Top 20 customers likely to increase spend if inventory improves:


,customer_id,total_spend,total_orders,total_items,constrained_spend,constrained_orders,constrained_items,constrained_share,avg_ticket,uplift_score
297,C00684,92897.24,1,3,65294.91,1.0,2.0,0.702872,92897.240,45258.982771
1294,C02859,74540.63,2,3,72216.58,2.0,2.0,0.968822,37270.315,39669.011117
246,C00580,86456.67,2,6,64601.45,2.0,3.0,0.747212,43228.335,35485.973418
494,C01122,47859.10,1,1,47859.10,1.0,1.0,1.000000,47859.100,33173.400229
1664,C03644,59262.15,2,2,59262.15,2.0,2.0,1.000000,29631.075,32553.063121
1052,C02363,46935.65,1,2,46249.70,1.0,1.0,0.985385,46935.650,32057.849157
1649,C03615,53156.50,1,2,46249.70,1.0,1.0,0.870067,53156.500,32057.849157
267,C00626,45270.60,1,1,45270.60,1.0,1.0,1.000000,45270.600,31379.188752
238,C00553,48506.90,1,2,43014.60,1.0,1.0,0.886773,48506.900,29815.448713
1627,C03568,80395.17,2,4,53896.60,2.0,3.0,0.670396,40197.585,29605.733539



Files saved:
 - customer_uplift_potential.csv  (all customers with uplift_score)
 - top_uplift_customers.csv       (top 20 high-potential customers)
